# How do we animate sequences of images?  

This notebook introduces some code to make animations to compare multiple images or plots. We apply it to the specific case of looking at the same spectra with different grating angles from the SBO 24" spectrograph; the animations we'll make may be really useful for figuring out which line is which when doing wavelength calibration!

## Make an animation of emission lamp + solar spectra
Let's make an animation showing the emission lamp spectra (and the scattered solar spectrum) for a few different grating angles. This may help us orient ourselves when we're looking at data from one particular grating angle. First we'll check what grating angles are available from the filenames...

In [1]:
# list the relevant HeNeAr files in our data directory 
!ls data/sbo/solar/Leto-Spectra-HeNeAr-*-20-10s-*.fits

data/sbo/solar/Leto-Spectra-HeNeAr-10-20-10s-18_32_37Z-00001.fits
data/sbo/solar/Leto-Spectra-HeNeAr-11-20-10s-18_36_22Z-00001.fits
data/sbo/solar/Leto-Spectra-HeNeAr-12-20-10s-18_39_58Z-00001.fits
data/sbo/solar/Leto-Spectra-HeNeAr-13-20-10s-18_43_09Z-00001.fits
data/sbo/solar/Leto-Spectra-HeNeAr-14-20-10s-18_46_54Z-00001.fits
data/sbo/solar/Leto-Spectra-HeNeAr-15-20-10s-18_52_12Z-00001.fits
data/sbo/solar/Leto-Spectra-HeNeAr-16-20-10s-18_55_11Z-00001.fits
data/sbo/solar/Leto-Spectra-HeNeAr-17-20-10s-18_58_30Z-00001.fits
data/sbo/solar/Leto-Spectra-HeNeAr-6-20-10s-18_17_54Z-00001.fits
data/sbo/solar/Leto-Spectra-HeNeAr-7-20-10s-18_21_57Z-00001.fits
data/sbo/solar/Leto-Spectra-HeNeAr-8-20-10s-18_25_11Z-00001.fits
data/sbo/solar/Leto-Spectra-HeNeAr-9-20-10s-18_28_13Z-00001.fits


Then we'll plot and animate them! The `with wri.saving` and `wri.grab_frame()` commands down below are doing most of the work of producing the animation; feel free to copy, paste, and modify this into your own code for your own animations!

In [ ]:
# import some useful tools
import glob
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as ani
from astropy.io import fits
from IPython.display import HTML
from exoatlas import one2another
import astropy.units as u
from specutils import Spectrum
from specutils.fitting import fit_generic_continuum
from astropy.modeling.models import Chebyshev1D

# allow units on astropy plots
from astropy.visualization import quantity_support
quantity_support();

<astropy.visualization.units.quantity_support.<locals>.MplQuantityConverter at 0x3147b3cb0>

In [3]:
# create a figure and animation writer
fi, ax = plt.subplots(5, 1, figsize=(8,6), sharex=True, constrained_layout=True)
wri = ani.writers['ffmpeg']()

# define some colors for each spectrum
colors = {'Sun':'black',
          'HeNeAr':'mediumvioletred', 
          'Ne':'red', 
          'HeAr':'indigo', 
          }

imshows = {}
plots = {}

# loop over forward and reverse directions
for direction in ['forward', 'backward']:
    # save an animation of the figure
    with wri.saving(fig=fi, outfile=f'animated-arc-lamps-{direction}.mp4', dpi=300):
        # loop over gratings
        for grating_angle in [7, 9, 11, 13]:
            # loop over elements
            for i, elements in enumerate(['HeAr', 'Ne', 'HeNeAr', 'Sun']):
                # load image file
                filename = glob.glob(f"data/sbo/solar/Leto-Spectra-{elements}-{grating_angle}-20-*-*.fits")[-1]
                hdu = fits.open(filename)
                image = hdu[0].data

                # extract a small region of it
                middle = int(image.shape[0]/2)
                halfwidth = 50
                region = image[middle-halfwidth:middle+halfwidth]

                # calculate a simple extracted spectrum
                extracted = np.mean(region, axis=0)
                w = np.arange(region.shape[1]) * u.pixel

                # remove continuum (from observing during the day)
                spectrum = Spectrum(flux=extracted * u.adu, spectral_axis=w)
                fitted_continuum = fit_generic_continuum(spectrum, median_window=25, model=Chebyshev1D(degree=3))
                if elements != 'Sun':
                    continuum = fitted_continuum(w).value
                    extracted = extracted - continuum
                    region = region - continuum[np.newaxis, :]

                # plot this element as image and line
                try:
                    # if plots already exist, just update them
                    imshows[elements].set_data(region)
                    plots[elements][0].set_ydata(extracted)
                except KeyError:
                    # make and store plots, if they don't exist
                    if elements == 'Sun':
                        vmax = 4e4
                    else:
                        vmax = 1e3
                    
                    # imshow the images
                    plt.sca(ax[i+1])
                    imshows[elements] = plt.imshow(region, 
                                                vmin=0,
                                                vmax=vmax, 
                                                aspect='auto', 
                                                cmap=one2another('white', colors[elements]))  
                    plt.ylabel(elements, color=colors[elements])

                    # plot the extracted lines
                    plt.sca(ax[0])
                    plots[elements] = plt.plot(extracted, color=colors[elements], alpha=0.3, label=elements)
                    plt.legend(frameon=False)
                    plt.ylim(0, vmax)
                    plt.title(f'Grating Angle = -{grating_angle}$^\circ$')
                    plt.sca(ax[-1])
                    plt.xlabel('Wavelength Pixel')
                    if direction == 'backward':
                        plt.xlim(len(w), 0)
            
            # grab this frame into the animation
            wri.grab_frame()

# don't show the last frame of the movie in the notebook 
plt.close()

You may notice that we plotted versions of the spectra with wavelength pixel going forward and wavelength pixel going backward. That's because for the SBO spectrograph, wavelength increases towards the left (= lower values of wavelength pixel). When trying to match up wavelengths and pixels, it's probably good to look at the spectra with wavelength pixel reversed!

In [4]:
from IPython.display import Video
Video("animated-arc-lamps-forward.mp4")

In [5]:
Video("animated-arc-lamps-backward.mp4")

Ta-da! Now we have some nice animations how spectra look different for different grating angles with the SBO 24" spectrograph.